# 01. Descubrimiento y registro de la API oficial de UNESCO UIS
Este notebook localiza la especificación pública de la API de UIS, registra la respuesta, calcula hashes SHA-256 y produce un diagnóstico reproducible. No modifica datos originales.

**Fuente oficial:** https://api.uis.unesco.org/api/public/documentation/


In [9]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys
import requests
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'uis'
META = ROOT / 'data' / 'metadata'
DIAG = ROOT / 'results' / 'diagnostics'
for p in (RAW, META, DIAG):
    p.mkdir(parents=True, exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({'User-Agent':'Haro-Morales-Education-Dynamics/0.1 (academic research)'})
print('ROOT:', ROOT)
print('Python:', sys.version.split()[0])
print('UTC:', datetime.now(timezone.utc).isoformat())

ROOT: C:\Users\ERICK\OneDrive\Documents\Tercer Semestre CDIA\ECUACIONES DIFERENCIALES\Articulo\education_dynamics_starter
Python: 3.12.16
UTC: 2026-11-06T00:29:42.345198+00:00


## 1. Probar ubicaciones estándar de la especificación OpenAPI
La documentación utiliza una interfaz web. Este bloque busca automáticamente el archivo JSON que describe los endpoints públicos.


In [10]:
base = 'https://api.uis.unesco.org'
candidates = [
    base + '/openapi.json',
    base + '/api/openapi.json',
    base + '/api/public/openapi.json',
    base + '/swagger/v1/swagger.json',
    base + '/api/swagger/v1/swagger.json',
    base + '/api/public/swagger/v1/swagger.json',
    base + '/api/public/documentation/openapi.json',
]
rows=[]
found=None
for url in candidates:
    try:
        r=SESSION.get(url, timeout=30)
        ctype=r.headers.get('content-type','')
        is_json='json' in ctype.lower()
        rows.append({'url':url,'status':r.status_code,'content_type':ctype,'bytes':len(r.content),'is_json':is_json})
        if r.ok and is_json and found is None:
            obj=r.json()
            if isinstance(obj,dict) and ('openapi' in obj or 'swagger' in obj or 'paths' in obj):
                found=(url,r,obj)
    except Exception as e:
        rows.append({'url':url,'status':'ERROR','content_type':'','bytes':0,'is_json':False,'error':repr(e)})
probe=pd.DataFrame(rows)
probe.to_csv(DIAG/'uis_openapi_probe.csv',index=False,encoding='utf-8-sig')
display(probe)
print('ESPECIFICACION ENCONTRADA:', found[0] if found else 'NO')

,url,status,content_type,bytes,is_json
0,https://api.uis.unesco.org/openapi.json,404,application/json; charset=utf-8,75,True
1,https://api.uis.unesco.org/api/openapi.json,404,application/json; charset=utf-8,79,True
2,https://api.uis.unesco.org/api/public/openapi....,404,application/json; charset=utf-8,86,True
3,https://api.uis.unesco.org/swagger/v1/swagger....,404,application/json; charset=utf-8,86,True
4,https://api.uis.unesco.org/api/swagger/v1/swag...,404,application/json; charset=utf-8,90,True
5,https://api.uis.unesco.org/api/public/swagger/...,404,application/json; charset=utf-8,97,True
6,https://api.uis.unesco.org/api/public/document...,200,text/html,767,False


ESPECIFICACION ENCONTRADA: NO


In [11]:
if found:
    url,r,spec=found
    raw_path=RAW/'uis_openapi_spec.json'
    raw_path.write_bytes(r.content)
    sha=hashlib.sha256(r.content).hexdigest()
    (META/'uis_openapi_spec.sha256').write_text(f'{sha}  {raw_path.name}\n',encoding='utf-8')
    paths=spec.get('paths',{})
    endpoint_rows=[]
    for path, methods in paths.items():
        for method, detail in methods.items():
            if method.lower() in {'get','post','put','delete','patch'}:
                endpoint_rows.append({'path':path,'method':method.upper(),'summary':detail.get('summary',''),'operationId':detail.get('operationId','')})
    endpoints=pd.DataFrame(endpoint_rows)
    endpoints.to_csv(META/'uis_api_endpoints.csv',index=False,encoding='utf-8-sig')
    print('SHA-256:',sha)
    print('ENDPOINTS:',len(endpoints))
    display(endpoints[endpoints.astype(str).apply(lambda s:s.str.contains('indicator|entit|geo|bulk',case=False,regex=True)).any(axis=1)].head(50))
else:
    print('No se localizo automaticamente la especificacion. El diagnostico quedo guardado para ajustar el siguiente paso sin inventar endpoints.')

No se localizo automaticamente la especificacion. El diagnostico quedo guardado para ajustar el siguiente paso sin inventar endpoints.


## 2. Registrar la página oficial de documentación


In [12]:
docs_url='https://api.uis.unesco.org/api/public/documentation/'
r=SESSION.get(docs_url,timeout=30)
r.raise_for_status()
html_path=RAW/'uis_api_documentation.html'
html_path.write_bytes(r.content)
html_sha=hashlib.sha256(r.content).hexdigest()
manifest=pd.DataFrame([{
    'source':'UNESCO UIS',
    'resource':'Public API documentation',
    'url':docs_url,
    'retrieved_at_utc':datetime.now(timezone.utc).isoformat(),
    'http_status':r.status_code,
    'content_type':r.headers.get('content-type',''),
    'bytes':len(r.content),
    'sha256':html_sha,
    'local_file':str(html_path.relative_to(ROOT))
}])
manifest.to_csv(META/'uis_source_manifest.csv',index=False,encoding='utf-8-sig')
display(manifest)
print('DOCUMENTACION REGISTRADA CORRECTAMENTE')

,source,resource,url,retrieved_at_utc,http_status,content_type,bytes,sha256,local_file
0,UNESCO UIS,Public API documentation,https://api.uis.unesco.org/api/public/document...,2026-11-06T00:29:43.942552+00:00,200,text/html; charset=UTF-8,767,c21d66acbed1b90aa59b70ff316717e8580de0e2fb12df...,data\raw\uis\uis_api_documentation.html


DOCUMENTACION REGISTRADA CORRECTAMENTE


## 3. Resultado del script
Si aparece `DOCUMENTACION REGISTRADA CORRECTAMENTE`, la fuente oficial quedó registrada. Si además aparece una especificación encontrada, el siguiente notebook podrá descargar el catálogo directamente. Si no aparece, se utilizará el BDDS oficial o se ajustará la ruta a partir del diagnóstico guardado.
